# 0) Imports

In [ ]:
import os
import sys
import glob
import json
import math
import time
import pdb
import pandas as pd
import numpy as np
import seaborn as sns
from tqdm import tqdm
from matplotlib import pyplot as plt
from pathlib import Path
from pathml.core import HESlide
from sklearn.utils import resample
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.model_selection import RepeatedKFold, RepeatedStratifiedKFold

base = Path('/path/to/project')
results = base.joinpath('results')
scripts= base.joinpath('scripts')
sys.path.append(scripts.joinpath('pancancer_he_classifier'))
from helpers import anno as annoHelper
%load_ext autoreload
%autoreload 2
data = base.joinpath('data')


# 1) Generate training tile_df dataframe

## 1) Collect Liver training tiles

In [ ]:
raw=data.joinpath('liver','aofei_liver_annotation')
sampleinfo = raw.joinpath('sampleinfo')
tiles= results.joinpath('liver_aofei','v9',
                        'tiles')
info = pd.read_csv(sampleinfo.joinpath('customized_sample_slides.tsv'), ## user is responseible to update this tsv file name accordingly
                   sep='\t')

all_dat = {'fn':[], 'slide':[], 'case':[], 'anno':[]}
version = 5 #Include case number
#Concatenate existing .tsv summaries:
tissue = 'liver'
all_tsv = pd.Series([str(x) for x in tiles.glob('*.tsv')])
missing = []
found = 0
for i,slide in enumerate(tqdm(info.slide.values)):
    slide=slide.split('.')[0]
    idx = all_tsv.str.contains(slide).values
    if any(idx):
        dat = pd.read_csv(all_tsv.loc[idx].values[0],sep='\t')
        tile_path = str(tiles.joinpath('224px',slide)) +'/'
        fns = tile_path + dat.tile 
        anno = dat.anno.copy()
        anno[anno.isna()] = 'notTumor'
        all_dat['fn'].extend(fns.values)
        all_dat['slide'].extend([i] * len(fns))
        all_dat['anno'].extend(anno.values)
        all_dat['case'].extend([slide.split('_')[0]] * len(fns))
        found += 1
    else:
        missing.append(slide)
n_missing = len(missing)
print('%d slides found, %d slides missing' % (found,n_missing))
print(missing)
    
tile_df = pd.DataFrame(all_dat)   
tile_df.loc[:,'tissue'] = tissue
tile_df.loc[:,'color_norm'] = True

fn = sampleinfo.joinpath('alltiles_df_v1_%dsamp_%dtiles.tsv' % (found,
                                                                tile_df.shape[0]))

print(fn)
tile_df.to_csv(fn,sep = '\t',index=False)
print(tile_df.shape)
tile_df.groupby('anno')['fn'].count()
tile_df.head(2)

In [ ]:
tile_df.groupby(['case','anno'])['fn'].count()

### Balance

In [ ]:
all_dat = pd.read_csv('/path/to/customized_sample_tiles.tsv',
                      sep = '\t')   ## user is responseible to update this tsv file name accordingly

## 2) Collect colorectal training tiles

In [ ]:
nct_crc_100k = data.joinpath('colorectal','NCT-CRC-HE-100K') #color normalized
colon_image = data.joinpath('colorectal','colon_image_sets') #all jpegs
print(nct_crc_100k.exists(), colon_image.exists())
all_dat = {'fn':[], 'slide':[], 'case':[], 'anno':[]}
tissue='colorectal'
slide = np.nan
all_fn = [x for x in nct_crc_100k.rglob('*.tif')]
for fn in tqdm(all_fn):
    if 'TUM' in fn.parent.parts[-1]:
        anno = 'Tumor'
    else:
        anno = 'notTumor'
    # if 'aca' in fn.parent.parts[-1]:
    #     anno = 'Tumor'
    # elif 'colon_n' in fn.parent.parts[-1]:
    #     anno = 'notTumor'
    all_dat['fn'].extend([str(fn)])
    all_dat['slide'].extend([''])
    all_dat['anno'].extend([anno])
    all_dat['case'].extend([''])

cr_df = pd.DataFrame(all_dat)                        
cr_df.loc[:,'tissue'] = tissue
cr_df.loc[:,'color_norm'] = True
fn = nct_crc_100k.joinpath('alltiles_df_v1_%dsamp_%dtiles.tsv' % (found,
                                                                cr_df.shape[0]))
print(fn)
cr_df.to_csv(fn,sep = '\t',index=False)
print(cr_df.shape)
cr_df.head(2)

In [ ]:
cr_df.groupby('anno')['fn'].count()

## 3) Collect lung training tiles

In [ ]:
lung_kaggle = data.joinpath('lung','lung_image_sets') #color normalized
print(lung_kaggle.exists())
all_dat = {'fn':[], 'slide':[], 'case':[], 'anno':[]}
tissue='lung'
slide = np.nan
all_fn = [x for x in lung_kaggle.rglob('*.jpeg')]
for fn in tqdm(all_fn):
    if 'lung_n' in fn.parent.parts[-1]:
        anno = 'notTumor'
    else:
        anno = 'Tumor'
    all_dat['fn'].extend([str(fn)])
    all_dat['slide'].extend([''])
    all_dat['anno'].extend([anno])
    all_dat['case'].extend([''])

lu_df = pd.DataFrame(all_dat)                        
lu_df.loc[:,'tissue'] = tissue
lu_df.loc[:,'color_norm'] = False
fn = lung_kaggle.joinpath('alltiles_df_v1_%dtiles.tsv' % (lu_df.shape[0]))
print(fn)
lu_df.to_csv(fn,sep = '\t',index=False)
print(lu_df.shape)
lu_df.head(2)

In [ ]:
lu_df.groupby('anno')['fn'].count()

## 4) Collect skin / melanoma training tiles

In [ ]:
raw=data.joinpath('skin','data','HE')
sampleinfo = data.joinpath('skin','sampleinfo')
tiles= results.joinpath('skin','v9',
                        'tiles')  ## user is responseible to update the file path accordingly
info = pd.read_csv(sampleinfo.joinpath('customized_sample_slides.tsv'),   ## user is responseible to update this tsv file name accordingly
                   sep='\t')

all_dat = {'fn':[], 'slide':[], 'case':[], 'anno':[]}
version = 5 #Include case number
#Concatenate existing .tsv summaries:
tissue = 'skin'
all_tsv = pd.Series([str(x) for x in tiles.glob('*.tsv')])
missing = []
found = 0
for i,slide in enumerate(tqdm(info.slide.values)):
    slide=slide.split('.')[0]
    idx = all_tsv.str.contains(slide).values
    if any(idx):
        dat = pd.read_csv(all_tsv.loc[idx].values[0],sep='\t')
        tile_path = str(tiles.joinpath('224px',slide)) +'/'
        fns = tile_path + dat.tile 
        anno = dat.anno.copy()
        anno[anno.isna()] = 'notTumor'
        all_dat['fn'].extend(fns.values)
        all_dat['slide'].extend([i] * len(fns))
        all_dat['anno'].extend(anno.values)
        all_dat['case'].extend([slide.split('_H&E')[0]] * len(fns))
        found += 1
    else:
        missing.append(slide)
n_missing = len(missing)
print('%d slides found, %d slides missing' % (found,n_missing))
print(missing)
    
tile_df = pd.DataFrame(all_dat)   
idx = tile_df.anno.values == 'anno'
tile_df.loc[idx,'anno'] = 'Tumor'
tile_df.loc[:,'tissue'] =  tissue
tile_df.loc[:,'color_norm'] = True

fn = sampleinfo.joinpath('alltiles_df_v1_%dsamp_%dtiles.tsv' % (found,
                                                                tile_df.shape[0]))

print(fn)
tile_df.to_csv(fn,sep = '\t',index=False)
print(tile_df.shape)
tile_df.groupby('anno')['fn'].count()
tile_df.head(2)

In [ ]:
sk_df.loc[sk_df.anno.values == 'anno','case'].unique()

## Last) Concatenate all together

In [ ]:
## user is responseible to update the file path and tsv file names accordingly
sk_df = pd.read_csv('/path/to/alltiles_df_sk.tsv', sep = '\t')
lv_df = pd.read_csv('/path/to/alltiles_df_lv.tsv',sep = '\t')
lu_df = pd.read_csv('/path/to/alltiles_df_lu.tsv',sep='\t')
cr_df = pd.read_csv('/path/to/alltiles_df_cr.tsv', sep = '\t')

In [ ]:
all_df = pd.concat((sk_df,lv_df,lu_df,cr_df),axis=0)
fn = data.joinpath('skin_lung_liver_colo_df_v1_%dtiles.tsv' % (all_df.shape[0]))  ## user is responseible to update this tsv file name accordingly
print(fn)
all_df.to_csv(fn,sep = '\t')
print('Finished')

In [ ]:
all_df = pd.read_csv('path/to/skin_lung_liver_colo_df_tiles.tsv',
                     sep = '\t')  ## user is responseible to update this tsv file name accordingly

In [ ]:
all_df.groupby(['tissue','anno'])['fn'].count()

In [ ]:
all_df.case.unique()

In [ ]:
all_df.groupby(['anno'])['fn'].count()

# 2) Create balanced tile set:

In [ ]:
all_df = pd.read_csv('/path/to/skin_lung_liver_colo_df_v1_tiles.tsv',
                     sep = '\t',
                    low_memory = False)  ## user is responseible to update this tsv file name accordingly
print('Finished')

In [ ]:
n_tumor_per_tissue = 14000
state = 36
collect = pd.DataFrame([])
for i,tissue in enumerate(all_df.tissue.unique()):
    idx = all_df.tissue == tissue
    cases = all_df.loc[idx,'case'].unique()
    #If cases are present, balance by cases
    if isinstance(cases[0],str):
        n_per_case = n_tumor_per_tissue // len(cases)
        print(tissue,n_per_case)
        for case in cases:
            idx2 = all_df.case.values == case
            keep = all_df.loc[idx & idx2,:].copy()
            t = keep.anno == 'Tumor'
            n = keep.anno == 'notTumor'
            if np.sum(t) < n_per_case:
                replace = True
            else:
                replace = False
            keep_t = resample(keep.loc[t,:],
                        n_samples=n_per_case,
                        random_state=state, 
                        replace=replace, )
            if np.sum(n) < n_per_case:
                replace = True
            else:
                replace = False
            keep_n = resample(keep.loc[n,:],
                       n_samples=n_per_case,
                        random_state=state, 
                        replace=replace, )
            collect = pd.concat((collect,keep_t,keep_n),axis=0)
    #If cases are not present, do not balance by case:
    else:
        print(tissue,n_tumor_per_tissue)
        keep = all_df.loc[idx,:].copy()
        t = keep.anno == 'Tumor'
        if np.sum(t) < n_tumor_per_tissue:
                replace = True
        else:
                replace = False
        n = keep.anno == 'notTumor'

        keep_t = resample(keep.loc[t,:],
                n_samples=n_tumor_per_tissue,
                random_state=state, 
                replace=replace, )
        if np.sum(n) < n_tumor_per_tissue:
                replace = True
        else:
                replace = False
        keep_n = resample(keep.loc[n,:],
                   n_samples=n_tumor_per_tissue,
                    random_state=state, 
                    replace=replace, )
        collect = pd.concat((collect,keep_t,keep_n),axis=0)
collect = collect.reset_index(drop=True)
fn = data.joinpath('balanced_%d_sk_lu_lv_cr_df_v1_%dtiles.tsv' % (n_tumor_per_tissue,collect.shape[0]))
print(fn)
collect.to_csv(fn,sep = '\t')
print('Finished')
collect.groupby(['tissue','anno'])['fn'].count()

## Test splitter for train on 3 cancer, test on 1 other:

In [ ]:
kfold = 4
splitter = GroupKFold(n_splits=kfold)
for slide_train_idx, slide_test_idx in splitter.split(collect.loc[:, 'fn'],
                                                      collect.loc[:, 'anno'],
                                                      groups=collect.loc[:, 'tissue']):
    print(collect.loc[slide_train_idx,:].groupby(['tissue','anno'])['fn'].count())

## Test splitter for: Train on all cancer, test on all cancer

In [ ]:
kfold = 10
reps = 1
i = 1
splitter = RepeatedKFold(n_splits = kfold,
                         n_repeats = reps,
                         random_state = state)
for slide_train_idx, slide_test_idx in splitter.split(collect.loc[:, 'fn'],
                                                      collect.loc[:, 'anno'],
                                                      ):
    print(i)
    print(collect.loc[slide_train_idx,:].groupby(['tissue','anno'])['fn'].count())
    i += 1